# 03. 실제 파형으로 AES 키 복구하기

TVLA에서는 누설 위치만 확인했습니다. 이제 알려진 평문과 키 후보로 AES 첫 라운드 중간값을 예측하고, 새로 측정한 파형 100개에서 16바이트 키를 복구합니다. 02에서 기록한 TinyAES 펌웨어를 그대로 사용하므로 다시 프로그래밍하지 않습니다. 02를 먼저 실행하십시오. 이전 측정에서는 약 10초가 걸렸습니다.

$$Z = SBox[P \oplus K]$$

공격자는 평문 `P`를 알고 있습니다. 키 후보 `K`를 하나씩 넣으면 중간값 `Z`를 예측할 수 있습니다.

In [ ]:
import chipwhisperer as cw
import importlib
import json
import matplotlib.pyplot as plt
import numpy as np

from cw_sca_tutorial.analysis import cpa_correlations, dpa_differences, recover_key_cpa
from cw_sca_tutorial.demo import repository_root
import cw_sca_tutorial.hardware as hardware

hardware = importlib.reload(hardware)
capture_aes_run = hardware.capture_aes_run

ROOT = repository_root()
devices = cw.list_devices()
assert len(devices) == 1, f"Expected one ChipWhisperer, found: {devices}"
run = capture_aes_run(
    ROOT / "firmware/prebuilt/simpleserial-aes-CWLITEARM.hex",
    trace_count=100,
    runs_root=ROOT / "runs",
    serial_number=devices[0]["sn"],
    platform_name="CWLITEARM",
    target_identification="STM32F303",
    program_firmware=False,
)
metadata = json.loads((run / "metadata.json").read_text())
data = np.load(run / "traces.npz")
traces = data["traces"]
plaintexts = data["plaintexts"]
known_key = data["key"]

print("DATA SOURCE : MEASURED")
print("RUN         :", run.name)
print("TARGET      :", metadata["target_identity"]["user_identification"])
print("TRACES      :", traces.shape)

## 실제 AES 파형

서로 다른 평문 10개의 파형을 겹쳐 봅니다. 전체 형태는 비슷하지만 데이터가 바뀌면서 각 샘플의 전력값이 조금씩 달라집니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(traces[:10].T, alpha=.35)
axes[0].set(title="Ten measured AES traces", xlabel="Sample", ylabel="Relative power")
axes[1].plot(traces.mean(axis=0), label="Mean")
axes[1].fill_between(
    np.arange(traces.shape[1]),
    traces.mean(axis=0) - traces.std(axis=0),
    traces.mean(axis=0) + traces.std(axis=0),
    alpha=.25,
    label="±1 standard deviation",
)
axes[1].set(title="Mean and trace variation", xlabel="Sample")
axes[1].legend()
plt.show()

## DPA로 첫 바이트 찾기

키 후보마다 `SBox[P XOR guess]`의 3번 비트를 예측합니다. 예측 비트가 0인 파형과 1인 파형의 평균을 빼면, 정답 후보에서 실제 누설 시점의 차이가 가장 크게 남습니다. 그래프는 첫 라운드 피크가 모인 800:2300 샘플을 확대합니다.

In [ ]:
differences = dpa_differences(traces, plaintexts, byte_index=0, bit=3)
dpa_scores = np.max(np.abs(differences), axis=1)
dpa_best = int(np.argmax(dpa_scores))
dpa_wrong = int(np.argsort(dpa_scores)[-2])

print(f"DPA best guess: 0x{dpa_best:02x}")
print(f"Known key byte: 0x{known_key[0]:02x}")
assert dpa_best == int(known_key[0])

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
dpa_window = slice(800, 2300)
image = axes[0].imshow(
    np.abs(differences[:, dpa_window]),
    aspect="auto",
    cmap="viridis",
    extent=[dpa_window.start, dpa_window.stop, 255, 0],
)
axes[0].axhline(dpa_best, color="white", linewidth=1)
axes[0].set(title="Measured DPA by key guess", xlabel="Sample", ylabel="Key guess")
fig.colorbar(image, ax=axes[0], label="|Mean difference|")
samples = np.arange(traces.shape[1])[dpa_window]
axes[1].plot(samples, differences[dpa_best, dpa_window], label=f"Correct 0x{dpa_best:02x}")
axes[1].plot(samples, differences[dpa_wrong, dpa_window], label=f"Best wrong 0x{dpa_wrong:02x}", alpha=.8)
axes[1].set(title="Correct and wrong DPA traces", xlabel="Sample")
axes[1].legend()
plt.show()

## CPA는 한 비트 대신 Hamming weight 전체를 사용합니다

DPA에서는 중간값의 한 비트만 사용했습니다. CPA에서는 `HW(SBox[P XOR guess])`의 0부터 8까지 값을 모두 사용하고, 예측값과 실제 파형의 Pearson 상관계수를 계산합니다. 같은 800:2300 구간에서 정답 후보의 상관 피크를 확인합니다.

In [ ]:
correlations = cpa_correlations(traces, plaintexts, byte_index=0)
cpa_scores = np.max(np.abs(correlations), axis=1)
cpa_best = int(np.argmax(cpa_scores))
cpa_wrong = int(np.argsort(cpa_scores)[-2])

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
cpa_window = slice(800, 2300)
image = axes[0].imshow(
    np.abs(correlations[:, cpa_window]),
    aspect="auto",
    cmap="magma",
    extent=[cpa_window.start, cpa_window.stop, 255, 0],
)
axes[0].axhline(cpa_best, color="cyan", linewidth=1)
axes[0].set(title="Measured CPA by key guess", xlabel="Sample", ylabel="Key guess")
fig.colorbar(image, ax=axes[0], label="|ρ|")
samples = np.arange(traces.shape[1])[cpa_window]
axes[1].plot(samples, np.abs(correlations[cpa_best, cpa_window]), label=f"Correct 0x{cpa_best:02x}")
axes[1].plot(samples, np.abs(correlations[cpa_wrong, cpa_window]), label=f"Best wrong 0x{cpa_wrong:02x}", alpha=.8)
axes[1].set(title="Correct and wrong CPA traces", xlabel="Sample", ylabel="|ρ|")
axes[1].legend()
plt.show()

print(f"CPA best guess: 0x{cpa_best:02x}")
print(f"Maximum correlation: {cpa_scores[cpa_best]:.4f}")

## 파형 수와 정답 순위

25개부터 500개까지 파형 수를 늘리며 첫 바이트 정답의 순위를 계산합니다. 파형이 늘어나면 가장 강한 오답의 상관은 낮아지고 정답 후보가 안정적으로 분리됩니다.

In [ ]:
counts = [25, 50, 75, 100]
true_scores, wrong_scores, ranks = [], [], []
for count in counts:
    corr = cpa_correlations(traces[:count], plaintexts[:count], byte_index=0)
    scores = np.max(np.abs(corr), axis=1)
    true = int(known_key[0])
    true_scores.append(scores[true])
    wrong_scores.append(np.max(np.delete(scores, true)))
    ranks.append(1 + int(np.sum(scores > scores[true])))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(counts, true_scores, "o-", label="Correct guess")
ax.plot(counts, wrong_scores, "o-", label="Best wrong guess")
for x, y, rank in zip(counts, true_scores, ranks):
    ax.annotate(f"rank {rank}", (x, y), xytext=(0, 7), textcoords="offset points", ha="center")
ax.set(title="Measured CPA convergence", xlabel="Number of traces", ylabel="Maximum |ρ|")
ax.legend()
plt.show()

## AES-128 전체 키

같은 CPA를 나머지 15바이트에도 적용합니다. 아래 두 줄이 같으면 16바이트를 모두 복구한 것입니다.

In [ ]:
from IPython.display import HTML, clear_output, display
import time

recovered, byte_scores, peak_samples = recover_key_cpa(traces, plaintexts)
assert np.array_equal(recovered, known_key)

for count in range(1, 17):
    visible = [f"{value:02x}" for value in recovered[:count]] + ["??"] * (16 - count)
    boxes = " ".join(
        f"<span style='display:inline-block;padding:8px 10px;margin:2px;border-radius:5px;"
        f"background:{'#e45756' if index < count else '#dddddd'};color:{'white' if index < count else '#666666'};"
        f"font-family:monospace;font-size:18px'>{value}</span>"
        for index, value in enumerate(visible)
    )
    clear_output(wait=True)
    display(HTML(
        f"<h3>Recovering AES-128 key: {count} / 16 bytes</h3>"
        f"<div>{boxes}</div>"
        f"<p style='font-family:monospace'>Current: {''.join(visible)}</p>"
    ))
    if count < 16:
        time.sleep(.18)

print("Expected :", bytes(known_key).hex())
print("Recovered:", bytes(recovered).hex())
print("Matching : 16 / 16 bytes")

## 바이트별 상관 피크

막대 높이는 각 바이트 정답 후보의 최대 상관계수입니다. 막대 위 숫자는 해당 피크가 나타난 샘플 위치입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(16), byte_scores, color="#4c78a8")
ax.set_xticks(range(16), [f"{value:02x}" for value in recovered])
ax.set(title="Full measured AES-128 key recovery", xlabel="Recovered key byte", ylabel="Maximum |ρ|")
for bar, sample in zip(bars, peak_samples):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), str(sample), ha="center", va="bottom", fontsize=8)
plt.show()